# Quick start - Sentinel-2 NDVI monthly via openEO

The headline openEO workflow: the `sentinel-2-l2a-ndvi-monthly` **recipe** builds a process graph (`load_collection -> mask_scl_dilation -> ndvi -> aggregate_temporal_period(month, mean) -> save`), the **CDSE backend executes it server-side**, and earthlens downloads the resulting NetCDF. We use a tiny bbox near Madrid and a one-month window so it stays small.

```bash
pip install earthlens[openeo]
```

## Setup

Imports up front: `os` / `pathlib.Path` for the credential probe, and the unified `EarthLens` entry point.

In [ ]:
import os
from pathlib import Path

from earthlens import EarthLens

## Credentials

openEO authenticates with CDSE via OIDC. The download cell below runs for real when credentials are present (env `OPENEO_CLIENT_ID` / `OPENEO_CLIENT_SECRET`, or a cached refresh token from a prior interactive login) and prints a skip note otherwise - so this notebook executes cleanly with or without secrets. See the Authentication page.

### Probing for a credential source

`has_openeo_credentials()` reports whether any OIDC credential source is available: headless client id/secret, a refresh token, or a cached `~/.openeo` login.

In [ ]:
def has_openeo_credentials() -> bool:
    """Whether some openEO OIDC credential source is available."""
    if os.environ.get("OPENEO_CLIENT_ID") and os.environ.get("OPENEO_CLIENT_SECRET"):
        return True
    if os.environ.get("OPENEO_REFRESH_TOKEN"):
        return True
    return (Path.home() / ".openeo").exists()

### A graceful-skip download wrapper

`run_or_skip()` runs the live `download()` when credentials exist and otherwise prints a skip note, so the notebook executes top-to-bottom with no errors whether or not CDSE OIDC credentials are configured.

In [ ]:
def run_or_skip(facade, **download_kwargs):
    """Run the live download when credentials exist, else print a skip note.

    Keeps the notebook executing top-to-bottom with no errors whether or not
    CDSE OIDC credentials are configured (so the docs build never needs secrets).
    """
    if not has_openeo_credentials():
        print(
            "No openEO OIDC credentials found - skipping the live download.\n"
            "Set OPENEO_CLIENT_ID / OPENEO_CLIENT_SECRET (headless) or sign in once\n"
            "interactively (see the Authentication page) to run this cell for real."
        )
        return []
    paths = facade.download(**download_kwargs)
    for path in paths:
        print(path)
    return paths

## Build the request

Name the openEO `dataset` recipe, the date window, the tiny Madrid bbox, and the output path. Printing `OUTPUT_KIND` confirms this is a file-writing backend.

In [ ]:
facade = EarthLens(
    data_source='openeo',
    dataset='sentinel-2-l2a-ndvi-monthly',
    variables=[],
    start='2023-06-01',
    end='2023-06-30',
    aoi=[3.67, 40.40, 3.72, 40.45],
    path='data/openeo-quickstart',
    max_cloud_cover=40,
)
print('output kind:', facade.datasource.OUTPUT_KIND)

## Download (server-side execution)

Hand the request to `run_or_skip()`: CDSE executes the process graph server-side and earthlens downloads the resulting NetCDF, returning the written paths.

In [ ]:
paths = run_or_skip(facade)
paths